<a href="https://colab.research.google.com/github/rishiks29/Retro_Project1/blob/main/Copy_of_PFBallotAI_Updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio

In [ ]:
import gradio as gr
import time

# ---------------- TIMER WITH STOP SUPPORT ----------------
stop_flag = False

def start_timer(minutes):
    global stop_flag
    stop_flag = False
    total = int(minutes * 60)

    for i in range(total, -1, -1):
        if stop_flag:
            yield "Stopped"
            return
        time.sleep(1)
        yield f"{i//60:02d}:{i%60:02d}"

def stop_timer():
    global stop_flag
    stop_flag = True
    return "Stopped"


# ---------------- TOPIC INFO FUNCTION (UPGRADED) ----------------
def topic_info(resolution):
    if not resolution.strip():
        return "Enter a PF resolution above to see background information, a brief summary, and common areas of clash."

    return f"""
## Topic Overview: What This Resolution Is About

**Resolution:** *{resolution}*

### 1. Brief Summary of the Topic
This resolution asks debaters to evaluate a major question about public policy, ethics, or global impact. At its core, the debate centers on:
- What the resolution *claims should happen*
- What the resolution *implies would happen*
- Whether those outcomes are beneficial, harmful, or uncertain

Your job as a judge is to understand the **central issue**, the **stakeholders**, and the **real-world consequences** being debated.

### 2. Core Clash You Should Expect
Most PF resolutions boil down to one or more of these tensions:
- **Benefits vs. Risks**
- **Short-term vs. Long-term impacts**
- **Individual rights vs. collective welfare**
- **Economic outcomes vs. social outcomes**
- **Security vs. liberty**
- **Feasibility vs. idealism**

### 3. What Pro Usually Argues
Pro teams typically argue that:
- The resolution leads to **positive outcomes**
- The benefits are **significant, likely, and widespread**
- The harms are **manageable or outweighed**

### 4. What Con Usually Argues
Con teams typically argue that:
- The resolution causes **serious harms**
- The risks are **too large or too likely**
- The benefits are **uncertain or outweighed**

### 5. What YOU Should Look For
- **Clarity**
- **Logic**
- **Evidence quality**
- **Clash**
- **Weighing**

### 6. How to Use This Section
Use this topic overview to:
- Ground yourself before flowing
- Understand the big picture
- Identify the main clash
- Track which team controls the most important impacts
"""


# ---------------- SUMMARY FUNCTION ----------------
def combined_summary(
    pro_const, con_const,
    pro_cf, con_cf,
    pro_reb, con_reb,
    pro_sum, con_sum,
    pro_gcf, con_gcf,
    pro_ff, con_ff
):
    return f"""
# Judge Summary (Organized by Round)

## Constructive
**Pro:** {pro_const}
**Con:** {con_const}

## Crossfire
**Pro:** {pro_cf}
**Con:** {con_cf}

## Rebuttal
**Pro:** {pro_reb}
**Con:** {con_reb}

## Summary
**Pro:** {pro_sum}
**Con:** {con_sum}

## Grand Crossfire
**Pro:** {pro_gcf}
**Con:** {con_gcf}

## Final Focus
**Pro:** {pro_ff}
**Con:** {con_ff}
"""


# ---------------- AUTO-SCORING FUNCTION ----------------
def auto_score(c, i, l, u):
    return round(c*0.25 + i*0.35 + l*0.25 + u*0.15, 2)


# ---------------- CSS ----------------
css = """
body {
    background: radial-gradient(circle at top left, #1e1e2f, #0f0f1a);
    color: white;
    font-family: 'Inter', sans-serif;
}

.header-bar {
    background: #0d0d15 !important;
    padding: 18px 20px !important;
    border-radius: 10px;
    margin-bottom: 20px;
}

.header-bar h1 {
    color: white !important;
    font-weight: 800 !important;
    margin: 0 !important;
    font-size: 2rem !important;
}

.header-bar h3 {
    color: #cccccc !important;
    font-weight: 500 !important;
    margin: 0 !important;
    font-size: 1.2rem !important;
}

.constructive, .crossfire, .rebuttal, .summary, .grandcf, .finalfocus {
    padding: 10px;
    border-radius: 10px;
    margin-bottom: 10px;
}

/* Styling for the Novice Tips Accordion */
.novice-tips-accordion {
    background: rgba(80,120,255,0.1); /* Slightly different color for variety */
    border-left: 4px solid rgba(80,120,255,0.4); /* Matching border */
    border-radius: 10px;
    margin-bottom: 20px;
    padding: 10px; /* Add some padding to the whole block */
}

.constructive { background: rgba(80,120,255,0.18); border-left: 4px solid rgba(80,120,255,0.55); }
.crossfire    { background: rgba(70,200,180,0.18); border-left: 4px solid rgba(70,200,180,0.55); }
.rebuttal     { background: rgba(255,180,60,0.18); border-left: 4px solid rgba(255,180,60,0.55); }
.summary      { background: rgba(160,120,255,0.18); border-left: 4px solid rgba(160,120,255,0.55); }
.grandcf      { background: rgba(255,110,80,0.18); border-left: 4px solid rgba(255,110,80,0.55); }
.finalfocus   { background: rgba(80,200,120,0.18); border-left: 4px solid rgba(80,200,120,0.55); }

button {
    background-color: #3b82f6 !important;
    color: white !important;
    border-radius: 10px !important;
}
"""


# ---------------- UI ----------------
with gr.Blocks() as app:

    # HEADER
    with gr.Group():
        gr.HTML("""
        <div class='header-bar'>
            <h1>PFJudgePro</h1>
            <h3>Built for novice PF judges — type what you hear, and the app will summarize and score it for you.</h3>
        </div>
        """, sanitize=False)

    # NOVICE TIPS
    with gr.Accordion("Novice Judge Tips (Click to Expand)", open=False, elem_classes="novice-tips-accordion"):
        gr.Markdown("""
### How to Judge Public Forum (PF) Debate — Quick Guide

- **Flow the round**
- **Evaluate clarity, logic, evidence, impact**
- **Look for clash**
- **Weigh impacts**
- **Judge only what is said in-round**
""")

    # TOPIC INPUT
    gr.Markdown("## Topic / Resolution")
    resolution = gr.Textbox(label="Enter PF Resolution")

    topic_output = gr.Markdown()
    resolution.change(topic_info, resolution, topic_output)

    # MAIN THREE-COLUMN LAYOUT
    with gr.Row():

        # LEFT COLUMN — ROUND NOTES
        with gr.Column(scale=3):

            gr.Markdown("## Round Notes")

            gr.Markdown("### Constructive", elem_classes=["constructive"])
            pro_const = gr.Textbox(lines=3, label="Pro Constructive")
            con_const = gr.Textbox(lines=3, label="Con Constructive")

            gr.Markdown("### Crossfire", elem_classes=["crossfire"])
            pro_cf = gr.Textbox(lines=3, label="Pro Crossfire")
            con_cf = gr.Textbox(lines=3, label="Con Crossfire")

            gr.Markdown("### Rebuttal", elem_classes=["rebuttal"])
            pro_reb = gr.Textbox(lines=3, label="Pro Rebuttal")
            con_reb = gr.Textbox(lines=3, label="Con Rebuttal")

            gr.Markdown("### Summary", elem_classes=["summary"])
            pro_sum = gr.Textbox(lines=3, label="Pro Summary")
            con_sum = gr.Textbox(lines=3, label="Con Summary")

            gr.Markdown("### Grand Crossfire", elem_classes=["grandcf"])
            pro_gcf = gr.Textbox(lines=3, label="Pro Grand Crossfire")
            con_gcf = gr.Textbox(lines=3, label="Con Grand Crossfire")

            gr.Markdown("### Final Focus", elem_classes=["finalfocus"])
            pro_ff = gr.Textbox(lines=3, label="Pro Final Focus")
            con_ff = gr.Textbox(lines=3, label="Con Final Focus")

        # MIDDLE COLUMN — SUMMARY + SCORING
        with gr.Column(scale=2):

            gr.Markdown("## Summary & Scoring")

            # COMBINED SUMMARY
            gr.Markdown("### Combined Summary (Round-by-Round)")
            summary_btn = gr.Button("Generate Combined Summary")
            summary_out = gr.Markdown()

            summary_btn.click(
                combined_summary,
                [
                    pro_const, con_const,
                    pro_cf, con_cf,
                    pro_reb, con_reb,
                    pro_sum, con_sum,
                    pro_gcf, con_gcf,
                    pro_ff, con_ff
                ],
                summary_out
            )

            # TEAM-SPECIFIC SCORING
            gr.Markdown("### Overall Debate Metrics (Team-Specific)")

            gr.Markdown("#### Pro Metrics")
            pro_clarity = gr.Slider(1, 5, value=3, label="Pro Clarity")
            pro_impact = gr.Slider(1, 5, value=3, label="Pro Impact")
            pro_likelihood = gr.Slider(1, 5, value=3, label="Pro Likelihood")
            pro_urgency = gr.Slider(1, 5, value=3, label="Pro Urgency")
            pro_score = gr.Number(label="Pro Score")

            gr.Markdown("#### Con Metrics")
            con_clarity = gr.Slider(1, 5, value=3, label="Con Clarity")
            con_impact = gr.Slider(1, 5, value=3, label="Con Impact")
            con_likelihood = gr.Slider(1, 5, value=3, label="Con Likelihood")
            con_urgency = gr.Slider(1, 5, value=3, label="Con Urgency")
            con_score = gr.Number(label="Con Score")

            score_btn = gr.Button("Calculate Scores")

            score_btn.click(
                lambda pc, pi, pl, pu, cc, ci, cl, cu: (
                    auto_score(pc, pi, pl, pu),
                    auto_score(cc, ci, cl, cu)
                ),
                [
                    pro_clarity, pro_impact, pro_likelihood, pro_urgency,
                    con_clarity, con_impact, con_likelihood, con_urgency
                ],
                [pro_score, con_score]
            )

        # RIGHT COLUMN — TIMER + SPEECH PRESETS
        with gr.Column(scale=1):

            gr.Markdown("## Round Timer")

            # Minutes selector moved here to fix timer error
            minutes = gr.Number(label="Minutes", value=4)

            timer_output = gr.Textbox(label="Timer")

            start_btn = gr.Button("Start Timer")
            stop_btn = gr.Button("Stop Timer")

            start_btn.click(start_timer, minutes, timer_output)
            stop_btn.click(stop_timer, outputs=timer_output)

            # SPEECH TIME PRESETS MOVED HERE
            gr.Markdown("### Speech Time Presets")
            speech_times = {
                "Constructive": 4,
                "Crossfire": 3,
                "Rebuttal": 4,
                "Summary": 3,
                "Grand Crossfire": 3,
                "Final Focus": 2,
                "Prep Time": 3
            }

            for label, mins in speech_times.items():
                btn = gr.Button(label)
                btn.click(lambda x=mins: x, outputs=minutes)

app.launch(css=css)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4fbaee1e891753663a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Deploying to Hugging Face Spaces

To deploy this Gradio application to Hugging Face Spaces, you will need two main files:
1.  `app.py`: This file will contain all your Python code for the Gradio application.
2.  `requirements.txt`: This file will list all the Python libraries your application needs (`gradio` in this case).

Follow these steps:

### Step 1: Create `app.py`
Run the following cell to create the `app.py` file with your current Gradio application code.

In [ ]:
app_code = """
import gradio as gr
import time

# ---------------- TIMER WITH STOP SUPPORT ----------------
stop_flag = False

def start_timer(minutes):
    global stop_flag
    stop_flag = False
    total = int(minutes * 60)

    for i in range(total, -1, -1):
        if stop_flag:
            yield "Stopped"
            return
        time.sleep(1)
        yield "{{i//60:02d}}:{{i%60:02d}}"

def stop_timer():
    global stop_flag
    stop_flag = True
    return "Stopped"


# ---------------- TOPIC INFO FUNCTION (UPGRADED) ----------------
def topic_info(resolution):
    if not resolution.strip():
        return "Enter a PF resolution above to see background information, a brief summary, and common areas of clash."

    return f\""\
## Topic Overview: What This Resolution Is About

**Resolution:** *{{resolution}}*

### 1. Brief Summary of the Topic
This resolution asks debaters to evaluate a major question about public policy, ethics, or global impact. At its core, the debate centers on:
- What the resolution *claims should happen*
- What the resolution *implies would happen*
- Whether those outcomes are beneficial, harmful, or uncertain

Your job as a judge is to understand the **central issue**, the **stakeholders**, and the **real-world consequences** being debated.

### 2. Core Clash You Should Expect
Most PF resolutions boil down to one or more of these tensions:
- **Benefits vs. Risks**
- **Short-term vs. Long-term impacts**
- **Individual rights vs. collective welfare**
- **Economic outcomes vs. social outcomes**
- **Security vs. liberty**
- **Feasibility vs. idealism**

### 3. What Pro Usually Argues
Pro teams typically argue that:
- The resolution leads to **positive outcomes**
- The benefits are **significant, likely, and widespread**
- The harms are **manageable or outweighed**

### 4. What Con Usually Argues
Con teams typically argue that:
- The resolution causes **serious harms**
- The risks are **too large or too likely**
- The benefits are **uncertain or outweighed**

### 5. What YOU Should Look For
- **Clarity**
- **Logic**
- **Evidence quality**
- **Clash**
- **Weighing**

### 6. How to Use This Section
Use this topic overview to:
- Ground yourself before flowing
- Understand the big picture
- Identify the main clash
- Track which team controls the most important impacts
\""\


# ---------------- SUMMARY FUNCTION ----------------
def combined_summary(
    pro_const, con_const,
    pro_cf, con_cf,
    pro_reb, con_reb,
    pro_sum, con_sum,
    pro_gcf, con_gcf,
    pro_ff, con_ff
):
    return f\""\
# Judge Summary (Organized by Round)

## Constructive
**Pro:** {{pro_const}}
**Con:** {{con_const}}

## Crossfire
**Pro:** {{pro_cf}}
**Con:** {{con_cf}}

## Rebuttal
**Pro:** {{pro_reb}}
**Con:** {{con_reb}}

## Summary
**Pro:** {{pro_sum}}
**Con:** {{con_sum}}

## Grand Crossfire
**Pro:** {{pro_gcf}}
**Con:** {{con_gcf}}

## Final Focus
**Pro:** {{pro_ff}}
**Con:** {{con_ff}}
\""\


# ---------------- AUTO-SCORING FUNCTION ----------------
def auto_score(c, i, l, u):
    return round(c*0.25 + i*0.35 + l*0.25 + u*0.15, 2)


# ---------------- CSS ----------------
css = \""\
body {
    background: radial-gradient(circle at top left, #1e1e2f, #0f0f1a);
    color: white;
    font-family: 'Inter', sans-serif;
}

.header-bar {
    background: #0d0d15 !important;
    padding: 18px 20px !important;
    border-radius: 10px;
    margin-bottom: 20px;
}

.header-bar h1 {
    color: white !important;
    font-weight: 800 !important;
    margin: 0 !important;
    font-size: 2rem !important;
}

.header-bar h3 {
    color: #cccccc !important;
    font-weight: 500 !important;
    margin: 0 !important;
    font-size: 1.2rem !important;
}

.constructive, .crossfire, .rebuttal, .summary, .grandcf, .finalfocus {
    padding: 10px;
    border-radius: 10px;
    margin-bottom: 10px;
}

/* Styling for the Novice Tips Accordion */
.novice-tips-accordion {
    background: rgba(80,120,255,0.1); /* Slightly different color for variety */
    border-left: 4px solid rgba(80,120,255,0.4); /* Matching border */
    border-radius: 10px;
    margin-bottom: 20px;
    padding: 10px; /* Add some padding to the whole block */
}

.constructive { background: rgba(80,120,255,0.18); border-left: 4px solid rgba(80,120,255,0.55); }
.crossfire    { background: rgba(70,200,180,0.18); border-left: 4px solid rgba(70,200,180,0.55); }
.rebuttal     { background: rgba(255,180,60,0.18); border-left: 4px solid rgba(255,180,60,0.55); }
.summary      { background: rgba(160,120,255,0.18); border-left: 4px solid rgba(160,120,255,0.55); }
.grandcf      { background: rgba(255,110,80,0.18); border-left: 4px solid rgba(255,110,80,0.55); }
.finalfocus   { background: rgba(80,200,120,0.18); border-left: 4px solid rgba(80,200,120,0.55); }

button {
    background-color: #3b82f6 !important;
    color: white !important;
    border-radius: 10px !important;
}
\""\


# ---------------- UI ----------------
with gr.Blocks() as app:

    # HEADER
    with gr.Group():
        gr.HTML("\""\
        <div class='header-bar'>
            <h1>PFJudgePro</h1>
            <h3>Built for novice PF judges — type what you hear, and the app will summarize and score it for you.</h3>
        </div>
        \""\", sanitize=False)

    # NOVICE TIPS
    with gr.Accordion("Novice Judge Tips (Click to Expand)", open=False, elem_classes="novice-tips-accordion"):
        gr.Markdown("\""\
### How to Judge Public Forum (PF) Debate — Quick Guide

- **Flow the round**
- **Evaluate clarity, logic, evidence, impact**
- **Look for clash**
- **Weigh impacts**
- **Judge only what is said in-round**
\""\")

    # TOPIC INPUT
    gr.Markdown("## Topic / Resolution")
    resolution = gr.Textbox(label="Enter PF Resolution")

    topic_output = gr.Markdown()
    resolution.change(topic_info, resolution, topic_output)

    # MAIN THREE-COLUMN LAYOUT
    with gr.Row():

        # LEFT COLUMN — ROUND NOTES
        with gr.Column(scale=3):

            gr.Markdown("## Round Notes")

            gr.Markdown("### Constructive", elem_classes=["constructive"])
            pro_const = gr.Textbox(lines=3, label="Pro Constructive")
            con_const = gr.Textbox(lines=3, label="Con Constructive")

            gr.Markdown("### Crossfire", elem_classes=["crossfire"])
            pro_cf = gr.Textbox(lines=3, label="Pro Crossfire")
            con_cf = gr.Textbox(lines=3, label="Con Crossfire")

            gr.Markdown("### Rebuttal", elem_classes=["rebuttal"])
            pro_reb = gr.Textbox(lines=3, label="Pro Rebuttal")
            con_reb = gr.Textbox(lines=3, label="Con Rebuttal")

            gr.Markdown("### Summary", elem_classes=["summary"])
            pro_sum = gr.Textbox(lines=3, label="Pro Summary")
            con_sum = gr.Textbox(lines=3, label="Con Summary")

            gr.Markdown("### Grand Crossfire", elem_classes=["grandcf"])
            pro_gcf = gr.Textbox(lines=3, label="Pro Grand Crossfire")
            con_gcf = gr.Textbox(lines=3, label="Con Grand Crossfire")

            gr.Markdown("### Final Focus", elem_classes=["finalfocus"])
            pro_ff = gr.Textbox(lines=3, label="Pro Final Focus")
            con_ff = gr.Textbox(lines=3, label="Con Final Focus")

        # MIDDLE COLUMN — SUMMARY + SCORING
        with gr.Column(scale=2):

            gr.Markdown("## Summary & Scoring")

            # COMBINED SUMMARY
            gr.Markdown("### Combined Summary (Round-by-Round)")
            summary_btn = gr.Button("Generate Combined Summary")
            summary_out = gr.Markdown()

            summary_btn.click(
                combined_summary,
                [
                    pro_const, con_const,
                    pro_cf, con_cf,
                    pro_reb, con_reb,
                    pro_sum, con_sum,
                    pro_gcf, con_gcf,
                    pro_ff, con_ff
                ],
                summary_out
            )

            # TEAM-SPECIFIC SCORING
            gr.Markdown("### Overall Debate Metrics (Team-Specific)")

            gr.Markdown("#### Pro Metrics")
            pro_clarity = gr.Slider(1, 5, value=3, label="Pro Clarity")
            pro_impact = gr.Slider(1, 5, value=3, label="Pro Impact")
            pro_likelihood = gr.Slider(1, 5, value=3, label="Pro Likelihood")
            pro_urgency = gr.Slider(1, 5, value=3, label="Pro Urgency")
            pro_score = gr.Number(label="Pro Score")

            gr.Markdown("#### Con Metrics")
            con_clarity = gr.Slider(1, 5, value=3, label="Con Clarity")
            con_impact = gr.Slider(1, 5, value=3, label="Con Impact")
            con_likelihood = gr.Slider(1, 5, value=3, label="Con Likelihood")
            con_urgency = gr.Slider(1, 5, value=3, label="Con Urgency")
            con_score = gr.Number(label="Con Score")

            score_btn = gr.Button("Calculate Scores")

            score_btn.click(
                lambda pc, pi, pl, pu, cc, ci, cl, cu: (
                    auto_score(pc, pi, pl, pu),
                    auto_score(cc, ci, cl, cu)
                ),
                [
                    pro_clarity, pro_impact, pro_likelihood, pro_urgency,
                    con_clarity, con_impact, con_likelihood, con_urgency
                ],
                [pro_score, con_score]
            )

        # RIGHT COLUMN — TIMER + SPEECH PRESETS
        with gr.Column(scale=1):

            gr.Markdown("## Round Timer")

            # Minutes selector moved here to fix timer error
            minutes = gr.Number(label="Minutes", value=4)

            timer_output = gr.Textbox(label="Timer")

            start_btn = gr.Button("Start Timer")
            stop_btn = gr.Button("Stop Timer")

            start_btn.click(start_timer, minutes, timer_output)
            stop_btn.click(stop_timer, outputs=timer_output)

            # SPEECH TIME PRESETS MOVED HERE
            gr.Markdown("### Speech Time Presets")
            speech_times = {
                "Constructive": 4,
                "Crossfire": 3,
                "Rebuttal": 4,
                "Summary": 3,
                "Grand Crossfire": 3,
                "Final Focus": 2,
                "Prep Time": 3
            }

            for label, mins in speech_times.items():
                btn = gr.Button(label)
                btn.click(lambda x=mins: x, outputs=minutes)

app.launch(css=css)
"""

with open("app.py", "w") as f:
    f.write(app_code)
print("app.py created successfully!")

app.py created successfully!


### Step 2: Create `requirements.txt`
Run the following cell to create the `requirements.txt` file.

In [ ]:
requirements_content = "gradio"

with open("requirements.txt", "w") as f:
    f.write(requirements_content)
print("requirements.txt created successfully!")

requirements.txt created successfully!


### Step 3: Deploy to Hugging Face Spaces

1.  **Go to Hugging Face Spaces:** Navigate to [https://huggingface.co/spaces/new](https://huggingface.co/spaces/new).
2.  **Create a New Space:**
    *   Choose a Space name.
    *   Select "Gradio" as the Space SDK.
    *   Choose "Public" or "Private" visibility as desired.
    *   Click "Create Space."
3.  **Upload Files:**
    *   Once your Space is created, you'll be taken to its page. Click on the "Files" tab.
    *   Click "Add file" and upload `app.py` and `requirements.txt` (which you just created in Colab) to the root directory of your Space.
4.  **Wait for Deployment:** Hugging Face Spaces will automatically detect your `app.py` and `requirements.txt` and start building your application. This might take a few minutes.
5.  **View Your App:** Once the build is complete, your Gradio app will be live and accessible from your Space page! You can embed it or share the link.